In [0]:
from delta.tables import DeltaTable
import pyspark.sql.functions as F
from pyspark.sql.window import Window

In [0]:
feat_columns = [
    # The Return Features
    'feat_daily_return', 
    'feat_highest_return', 
    'feat_lowest_return',
    
    # The New "Orthogonal" Features (Context)
    'feat_volatility_ratio',   # Is the market nervous?
    'feat_relative_Volume',    # Is volume unusually high?
    'feat_intraday_return',    # Did it move a lot during the day?
    'feat_pivot_point',

    # The Candlestick Shapes
    'feat_upper_shadow',       # Rejection from highs
    'feat_Lower_shadow'        # Rejection from lows
]
needed_columns = ['Date', 'company', 'label', 'day_of_week', 'day_of_month'] + feat_columns
stocks_w_features = spark.table(f"stocks.silver.stocks_w_features").select(needed_columns)


In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# 1. Define your dataframe
# df_raw = spark.table("your_catalog.your_schema.your_table")

# 2. Identify your columns
id_col = "company"
date_col = "Date"
# List of columns you want to fill (everything except ID and Date)
data_cols = [c for c in stocks_w_features.columns if c not in [id_col, date_col]]

# 3. Create the Calendar Grid (Same as before)
min_max = stocks_w_features.select(F.min(date_col).alias("min_d"), F.max(date_col).alias("max_d")).collect()[0]

df_dates = spark.sql(f"""
    SELECT explode(sequence(
        to_date('{min_max['min_d']}'), 
        to_date('{min_max['max_d']}'), 
        interval 1 day
    )) as full_date
""")

# 4. Cross Join to get (Company x Date)
df_companies = stocks_w_features.select(id_col).distinct()
df_full_grid = df_companies.crossJoin(df_dates)

# 5. Join back to raw data
# alias "grid" = the complete calendar (no missing rows)
# alias "raw" = your original data (has missing rows/holidays)
df_joined = df_full_grid.alias("grid").join(
    stocks_w_features.alias("raw"), 
    (F.col(f"grid.{id_col}") == F.col(f"raw.{id_col}")) & 
    (F.col("grid.full_date") == F.col(f"raw.{date_col}")), 
    "left"
)

# 6. Define the Window for Forward Filling
window_spec = Window.partitionBy(f"grid.{id_col}").orderBy("grid.full_date").rowsBetween(Window.unboundedPreceding, 0)

# 7. Dynamically generate fill expressions for ALL data columns
# This loops through 'label', 'feature1', 'feature2', etc. and creates a filled version
fill_expressions = [
    F.last(f"raw.{c}", ignorenulls=True).over(window_spec).alias(c) 
    for c in data_cols
]

# 8. Final Select
# We take ID and Date from the GRID (to ensure no nulls)
# And we take the rest from our filled expressions
df_ready_for_automl = df_joined.select(
    F.col(f"grid.{id_col}").alias(id_col),
    F.col("grid.full_date").alias(date_col),
    *fill_expressions  # <--- This unpacks and adds all your other columns
)

# 9. Save
# df_ready_for_automl.write.mode("overwrite").saveAsTable("stocks_gold.clean_forecasting_data")

In [0]:
df_ready_for_automl.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("stocks.gold.df_ready_for_automl")

In [0]:

windowSpec = Window.partitionBy("company").orderBy("Date")
gold_df = stocks_w_features
for column in feat_columns:

    # .withColumn(f"prev_{column}_4", F.lag(f"{column}", 4).over(windowSpec)) \
    # .withColumn(f"prev_{column}_5", F.lag(f"{column}", 5).over(windowSpec)) \
    gold_df = gold_df \
    .withColumn(f"prev_{column}_1", F.lag(f"{column}", 1).over(windowSpec)) \
    .withColumn(f"prev_{column}_2", F.lag(f"{column}", 2).over(windowSpec)) \
    .withColumn(f"prev_{column}_3", F.lag(f"{column}", 3).over(windowSpec)) \
    # .na.drop() # Drop rows with nulls (start of history)

gold_df = gold_df.drop(*feat_columns)

In [0]:
gold_df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("stocks.gold.stocks_w_prev_returns")

In [0]:
%sql
select * from 
-- stocks.gold.stocks_w_prev_returns
stocks.gold.stocks_w_prev_returns
order by company, date desc